# R4 · P3, P4 y P7 (cada respuesta: número, gráfico y 2 o 3 líneas)

**Responsable:** Dan (Rol B)  
**Revisor:** Leo (Rol D)

## Qué queda resuelto

- [x] P3 · Letalidad aparente mensual frente al % de vacunados (Chile y 3 países); advertencia del testeo
- [x] P4 · Índice de calidad: los 10 peores, por qué y qué se decidió
- [x] P7 · Pregunta propia con la fuente externa (correlación frente a causalidad)

In [ ]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
import duckdb
import pandas as pd
from IPython.display import Image, display

DW_PATH = RAIZ / 'data' / 'dw_covid.duckdb'

## P3 · Letalidad aparente mensual frente al % de vacunados (Chile y 3 países); advertencia del testeo

In [ ]:
con = duckdb.connect(str(DW_PATH), read_only=True)
paises_p3 = ['Chile', 'Argentina', 'Peru', 'Uruguay']
ph3 = ','.join([f"'{p}'" for p in paises_p3])
p3 = con.execute(f'''
    WITH cov_mes AS (
        SELECT t.country_region, d.anio_mes,
               SUM(GREATEST(f.casos_nuevos, 0)) AS casos_mes,
               SUM(GREATEST(f.fallecidos_nuevos, 0)) AS muertes_mes
        FROM fact_covid f
        JOIN dim_territorio t ON f.territorio_key = t.territorio_key
        JOIN dim_fecha d ON f.fecha_key = d.fecha_key
        WHERE t.country_region IN ({ph3}) AND d.fecha BETWEEN '2020-04-01' AND '2022-12-31'
        GROUP BY 1, 2
    ),
    vac_mes AS (
        SELECT t.country_region, d.anio_mes, MAX(v.pct_vacunacion_completa) AS pct_vac
        FROM fact_vacunacion v
        JOIN dim_fecha d ON v.fecha_key = d.fecha_key
        JOIN (SELECT DISTINCT country_region, iso3 FROM dim_territorio WHERE iso3 IS NOT NULL) t ON v.iso3 = t.iso3
        WHERE t.country_region IN ({ph3})
        GROUP BY 1, 2
    )
    SELECT c.country_region, c.anio_mes,
           ROUND(c.muertes_mes * 100.0 / NULLIF(c.casos_mes, 0), 2) AS letalidad_pct,
           COALESCE(v.pct_vac, 0) AS pct_vac
    FROM cov_mes c LEFT JOIN vac_mes v ON c.country_region = v.country_region AND c.anio_mes = v.anio_mes
''').df()
for pais in paises_p3:
    sub = p3[p3['country_region'] == pais]
    print(pais, '| <50% vac:', round(sub[sub['pct_vac'] < 50]['letalidad_pct'].median(), 2), '% -> >=50% vac:', round(sub[sub['pct_vac'] >= 50]['letalidad_pct'].median(), 2), '%')
display(Image(filename=str(RAIZ / 'docs' / 'figuras' / 'p3_letalidad_vacunacion.png')))

## P4 · Índice de calidad: los 10 peores, por qué y qué se decidió

In [ ]:
indice = pd.read_csv(RAIZ / 'data' / 'clean' / 'indice_calidad_paises.csv')
display(indice.head(10))
print('Decisión: no se imputa ni elimina; se marca en DW y se usa media móvil 7 días (ver docs/decisiones_calidad.md).')

## P7 · Pregunta propia con la fuente externa (correlación frente a causalidad)

In [ ]:
p7 = con.execute('''
    WITH pob AS (
        SELECT country_region, MAX(iso3) AS iso3,
               COALESCE(MAX(CASE WHEN nivel_granularidad = 'pais' THEN population END),
                        SUM(CASE WHEN nivel_granularidad = 'region_estado' THEN population END),
                        SUM(population)) AS population
        FROM dim_territorio WHERE NOT es_entidad_especial AND iso3 IS NOT NULL
        GROUP BY 1 HAVING population >= 1000000
    ),
    vac_2021 AS (SELECT iso3, MAX(pct_vacunacion_completa) AS vac_dic_2021 FROM fact_vacunacion WHERE fecha_key <= 20211231 GROUP BY 1),
    muertes_2022 AS (
        SELECT t.country_region, SUM(GREATEST(f.fallecidos_nuevos, 0)) AS muertes_2022
        FROM fact_covid f JOIN dim_territorio t ON f.territorio_key = t.territorio_key JOIN dim_fecha d ON f.fecha_key = d.fecha_key
        WHERE d.anio = 2022 AND NOT t.es_entidad_especial GROUP BY 1
    )
    SELECT p.country_region, ROUND(v.vac_dic_2021, 1) AS vac_dic_2021,
           ROUND(m.muertes_2022 * 100000.0 / p.population, 1) AS mortalidad_2022_100k
    FROM pob p JOIN vac_2021 v ON p.iso3 = v.iso3 JOIN muertes_2022 m ON p.country_region = m.country_region
    WHERE v.vac_dic_2021 IS NOT NULL
''').df()
print('n =', len(p7), '| Correlación Pearson r =', round(p7['vac_dic_2021'].corr(p7['mortalidad_2022_100k']), 3))
con.close()
display(Image(filename=str(RAIZ / 'docs' / 'figuras' / 'p7_vacunacion_vs_mortalidad.png')))